# This page helps you prepare your environment for the Anthropics Bootcamp.

## Test This in VS Code

1. Open this notebook in VS Code and select a Python 3.13+ kernel from the picker in the top-right.
2. Run the setup cell below with the Run Cell button or `Shift+Enter`.
3. If the cell creates a `.env` file, open it, paste your `ANTHROPIC_API_KEY`, save the file, and run the setup cell again.
4. The setup is working when the cell reports Python, dependency, and Node.js checks as successful and shows the message `API key verified - you're connected to Claude.`

In [4]:
# ── Install & Import ──
# Install dependencies into THIS kernel — safe to re-run; survives locked-down (PEP 668) Pythons.
import importlib.util, subprocess, sys

def _verify_python_version(min_major=3, min_minor=13):
    """Require Python >= min_major.min_minor in the active kernel."""
    current = sys.version_info[:3]
    if current < (min_major, min_minor, 0):
        raise SystemExit(
            f"Python {min_major}.{min_minor}+ is required for this bootcamp. "
            f"Current kernel is {current[0]}.{current[1]}.{current[2]}. "
            "Select a Python 3.13+ interpreter/kernel in VS Code and re-run this cell."
        )
    print(f"✓ Python version verified ({current[0]}.{current[1]}.{current[2]})")

_verify_python_version()

def _ensure_packages(requirements):
    """requirements: list of (import_name, pip_spec). Install only what is missing,
    into the running interpreter. Tries a normal install, then user-space, then a
    
    PEP 668 override (user-space first, system-wide only as a last resort). Every
    attempt is silent — pip's output is captured, not streamed — so a locked-down
    Python (Homebrew or Debian, PEP 668) no longer dumps a scary
    'externally-managed-environment' wall of text when a fallback is what actually
    succeeds. Only if every strategy fails does it surface the reason, with the
    venv fix instead of a raw traceback."""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " — first run only, please wait…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "(no output from pip)"
    raise SystemExit(
        "\n  Couldn't install: " + ", ".join(missing) + "\n"
        "  This Python is locked down (PEP 668) or offline. Quickest fix is a venv:\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windows: see SETUP.md\n"
        f"      pip install {' '.join(missing)}\n"
        "  Then pick the .venv interpreter in VS Code (kernel picker, top-right) and Run All.\n"
        "  Corporate proxy or PyPI blocked? See SETUP.md in the repo root.\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ Dependencies ready")

import anthropic
import json
import time
import os

# ── Node.js Verification ──
def _verify_node_js():
    """Verify Node.js is installed and can execute a command."""
    try:
        version = subprocess.run(["node", "--version"], capture_output=True, text=True)
    except FileNotFoundError:
        raise SystemExit(
            "Node.js was not found on PATH. Install Node.js and re-run this cell."
        )
    if version.returncode != 0:
        details = (version.stderr or version.stdout or "").strip()
        raise SystemExit(
            "Node.js appears installed but did not respond correctly to --version."
            + (f" Details: {details}" if details else "")
        )

    probe = subprocess.run(
        ["node", "-e", "console.log('node-check-ok')"],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0 or "node-check-ok" not in (probe.stdout or ""):
        details = (probe.stderr or probe.stdout or "").strip()
        raise SystemExit(
            "Node.js is installed, but a test command failed to execute."
            + (f" Details: {details}" if details else "")
        )

    print(f"✓ Node.js verified ({(version.stdout or '').strip()})")

_verify_node_js()

# ── API Key Configuration ──
import os

def _status(ok, msg):
    """Green/red banner in notebooks; plain text when run as a script."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os, pathlib

# ── API key via .env (gitignored — never committed) ──
# Your key lives in a .env file. We look for one next to this notebook or in any parent
# folder, so a single .env at the repo root can serve every exercise (paste once). If none
# exists yet, we create one from this template — fill it in, save, and re-run the cell.
_ENV_TEMPLATE = (
    "# Paste your Anthropic API key after the = (no quotes, no spaces), then save\n"
    "# and re-run the setup cell. Get a key at https://console.anthropic.com/\n"
    "# This file is gitignored — your key is never committed.\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
)

def _resolve_env_file():
    """Nearest existing .env walking up from the working dir (so one root .env serves every
    exercise); if none exists yet, point at the repo root — or this folder if the notebook
    was opened on its own."""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"

_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, paste your key after "
          "ANTHROPIC_API_KEY=, save, then re-run this cell.")

# Tiny .env parser (no python-dotenv dependency). Re-read on every run, so pasting your
# key and re-running picks it up. A real key in the environment (shell / Claude Code / CI)
# wins; the placeholder never sticks.
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)
_shell = os.environ.get("ANTHROPIC_API_KEY", "").strip()
api_key = _shell if _shell.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
if not api_key.startswith("sk-ant-"):
    print(
        "\n📋 Add your key to continue:\n"
        f"   1. Open this file:  {_env_file}\n"
        "   2. Replace  paste-your-key-here  with your key (it starts with sk-ant-)\n"
        "   3. Save the file, then click ▶ on this cell again.\n"
    )
else:
    import anthropic
    client = anthropic.Anthropic(api_key=api_key, timeout=30.0, max_retries=1)
    try:
        client.messages.create(model="claude-haiku-4-5", max_tokens=1,
                               messages=[{"role": "user", "content": "ping"}])
    except anthropic.AuthenticationError:
        _status(False, "That key was rejected. Run this cell again and paste the whole key (it starts with sk-ant-).")
        raise SystemExit("API key not accepted - re-run this cell and try again.")
    except Exception as exc:
        _status(False, "Could not reach the Claude API (" + type(exc).__name__ + "). Check your connection, then run this cell again.")
        raise
    else:
        os.environ["ANTHROPIC_API_KEY"] = api_key  # later cells (and any !python commands) pick it up from here
        _status(True, "API key verified - you're connected to Claude.")

    client = anthropic.Anthropic(timeout=900.0)  # Longer timeout: needed for max_tokens>21333 with non-streaming calls
    MODEL = "claude-sonnet-4-6"

✓ Python version verified (3.13.5)
✓ Dependencies ready
✓ Node.js verified (v24.12.0)
